# 🎨 Image Generator Starter

**AI Learning Playground — Educational Quickstart Blueprint**

Build, run, and deploy a text-to-image generator
using SDXL-Turbo — a fast diffusion model that produces high-quality images in just 4 steps.

---

## What This Notebook Covers

| Step | Topic | Key Concept |
|------|-------|-------------|
| 1 | Environment Setup | Installing dependencies, verifying GPU + CUDA memory |
| 2 | Configure Settings | Loading `image_gen.yaml`, resolving `image_model_path` |
| 3 | Initialize Model | Instantiating `ImageGenModel` (pipeline loads lazily) |
| 4 | Demo | Calling `model.predict()` → base64 image → display in notebook |
| 5 | GPU Monitoring | VRAM usage during diffusion pipeline |
| 6 | Register Model | Logging to MLflow as `AIStudio-EQ-ImageGen` |
| 7 | Verify | Loading registered model and running a test inference |

## How SDXL-Turbo Works

```
Text Prompt
     ↓  Text encoder (CLIP)
Text embeddings
     ↓  UNet (4 denoising steps)
Latent representation
     ↓  VAE decoder
PIL Image  →  base64 PNG  →  model.predict() output
```

SDXL-Turbo uses **adversarial diffusion distillation** to compress 50 standard steps into 4,
maintaining quality while drastically reducing inference time.

In [ ]:
import sys
import time

sys.path.insert(0, "..")

start_time = time.time()
print("⏱️  Notebook started")

## 1. Environment Setup

Install all required packages and verify GPU availability.
SDXL-Turbo requires a GPU with at least 8 GB VRAM.

In [ ]:
%pip install -q -r ../requirements.txt

import torch

cuda_available = torch.cuda.is_available()
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {'✅ Available' if cuda_available else '❌ Not found — GPU is required for SDXL-Turbo'}")

if cuda_available:
    props = torch.cuda.get_device_properties(0)
    print(f"GPU      : {props.name}")
    vram_gb = props.total_memory / 1e9
    print(f"VRAM     : {vram_gb:.1f} GB {'✅' if vram_gb >= 8 else '⚠️ (8 GB recommended)'}")
    print(f"Compute  : {props.major}.{props.minor}")

## 2. Configure Settings

Load `configs/image_gen.yaml` which specifies `capability: image_gen`
and the path to the SDXL-Turbo model directory.

In [ ]:
import os
from src.utils import load_config

config = load_config("../configs/image_gen.yaml")

image_model_path = config.get("image_model_path", "/home/jovyan/datafabric/sdxl-turbo")

print(f"Capability       : {config.get('capability')}")
print(f"Image model path : {image_model_path}")
print(f"Model exists     : {'✅' if os.path.exists(image_model_path) else '❌ Not found'}")
print(f"UI mode          : {config.get('ui', {}).get('mode', 'streamlit')}")

## 3. Verify Assets

Confirm the SDXL-Turbo model directory is present before loading.

In [ ]:
from src.utils import log_asset_status

assets = [
    {"name": "SDXL-Turbo model",  "path": image_model_path,               "required": True},
    {"name": "Config YAML",       "path": "../configs/image_gen.yaml",     "required": True},
    {"name": "Image gen demo UI", "path": "../demo/image_gen/main.py",     "required": False},
]

log_asset_status(assets)

## 4. Initialize ImageGenModel

Instantiate `ImageGenModel` — the same class registered in MLflow.

**Lazy loading design:**
The diffusion pipeline (several GB in memory) is NOT loaded in `__init__`.
It is loaded on the first `predict()` call and cached for subsequent calls.
This keeps initialization fast and avoids loading unused models.

In [ ]:
from src.mlflow.models.image_gen import ImageGenModel

print("Initializing ImageGenModel...")
print("(Pipeline loads lazily on first predict() call)")

model = ImageGenModel(
    config=config,
)

print("\n✅ ImageGenModel ready")
print(f"   Image model path : {model.image_model_path}")

## 5. Demo: Text-to-Image Generation

Call `model.predict()` with a text prompt and display the generated image inline.

The `ImageGenModel` accepts one column:
- `prompt` — text description of the image to generate

The `answer` output is a base64-encoded PNG string — we decode it to display it.

In [ ]:
import base64
import pandas as pd
from IPython.display import Image as IPImage, display

def generate_and_display(prompt: str) -> None:
    """Generate an image and display it inline in the notebook."""
    print(f"Generating: '{prompt[:60]}...'")
    t0 = __import__('time').time()
    
    result = model.predict(pd.DataFrame([{"prompt": prompt}]))
    elapsed = __import__('time').time() - t0
    answer = result["answer"].iloc[0]
    
    if len(answer) > 200 and not answer.startswith("❌"):
        img_bytes = base64.b64decode(answer)
        print(f"✅ Generated in {elapsed:.1f}s")
        display(IPImage(data=img_bytes, width=512))
    else:
        print(f"Error: {answer}")

# First generation — this also loads the pipeline into VRAM
generate_and_display(
    "A futuristic AI robot teaching in a university classroom, photorealistic, warm lighting, high detail"
)

In [ ]:
# Second generation — pipeline already in VRAM, much faster
generate_and_display(
    "Abstract visualization of a neural network, glowing blue nodes and connections, dark background, digital art"
)

In [ ]:
import time
import plotly.graph_objects as go

# Measure generation time across multiple prompts (pipeline already warm)
bench_prompts = [
    "A scenic mountain landscape at sunrise",
    "Abstract data visualization with colorful charts",
    "A circuit board with glowing components, macro photography",
]

times = []
for p in bench_prompts:
    t0 = time.time()
    model.predict(pd.DataFrame([{"prompt": p}]))
    times.append(time.time() - t0)

fig = go.Figure(go.Bar(
    x=[f"Prompt {i+1}" for i in range(len(bench_prompts))],
    y=times,
    marker_color="#0096d6",
    text=[f"{t:.1f}s" for t in times],
    textposition="auto",
))
fig.update_layout(
    title="SDXL-Turbo Generation Time per Prompt (4 steps)",
    xaxis_title="Prompt",
    yaxis_title="Time (seconds)",
    template="plotly_dark",
)
fig.show()

## 6. GPU Monitoring

SDXL-Turbo uses significantly more VRAM than the chatbot (full UNet + VAE loaded).
Observe how the diffusion pipeline occupies GPU memory.

In [ ]:
from src.gpu_monitor import GPUMonitor

monitor = GPUMonitor()
monitor.display_dashboard()

## 7. Register with MLflow

Register this model as **`AIStudio-EQ-ImageGen`** — independently from the chatbot model.

The `configs/image_gen.yaml` has `capability: image_gen` so `loader.py` will
automatically select `ImageGenModel` when this registered model is loaded or served.

In [ ]:
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")

ARTIFACT_PATH = "AIStudio-EQ-ImageGen"
MODEL_NAME    = "AIStudio-EQ-ImageGen"

print(f"Artifact path  : {ARTIFACT_PATH}")
print(f"Registered as  : {MODEL_NAME}")
print(f"Tracking URI   : {mlflow.get_tracking_uri()}")

In [ ]:
from mlflow.models import ModelSignature
from mlflow.types.schema import ColSpec, Schema

# Minimal input schema: ImageGenModel only needs the prompt
input_schema = Schema([
    ColSpec("string", "prompt"),   # Text description of the image
])

output_schema = Schema([
    ColSpec("string", "answer"),    # Base64-encoded PNG image
    ColSpec("string", "messages"),  # JSON metadata
])

signature = ModelSignature(inputs=input_schema, outputs=output_schema)

print("Input  : prompt (string)")
print("Output : answer (base64 PNG), messages (JSON metadata)")

In [ ]:
from src.mlflow.logger import Logger

with mlflow.start_run(run_name=f"register-{ARTIFACT_PATH}") as run:
    Logger.log_model(
        signature     = signature,
        artifact_path = ARTIFACT_PATH,
        config_path   = "../configs/image_gen.yaml",
        model_path    = image_model_path,    # Patches image_model_path into config copy
        demo_folder   = "../demo/image_gen",
    )
    run_id = run.info.run_id

print(f"✅ Model logged | Run ID: {run_id}")

model_uri = f"runs:/{run_id}/{ARTIFACT_PATH}"
reg = mlflow.register_model(model_uri=model_uri, name=MODEL_NAME)

print(f"✅ Registered  : {MODEL_NAME} v{reg.version}")

## 8. Verify Registration

Load the registered model and confirm it generates an image successfully.

In [ ]:
loaded_model = mlflow.pyfunc.load_model(model_uri=model_uri)

test_result = loaded_model.predict(pd.DataFrame([{
    "prompt": "A simple geometric pattern, blue and white, minimalist"
}]))

answer = test_result["answer"].iloc[0]
if len(answer) > 200 and not answer.startswith("❌"):
    img_bytes = base64.b64decode(answer)
    print("✅ Registered model generated an image successfully")
    display(IPImage(data=img_bytes, width=256))
else:
    print(f"Response: {answer}")

In [ ]:
elapsed = time.time() - start_time
print(f"⏱️  Total notebook time: {elapsed:.1f}s ({elapsed / 60:.1f} min)")

---

## ✅ What We Accomplished

| Step | Result |
|------|--------|
| Environment | CUDA verified, VRAM checked |
| ImageGenModel | Initialized with SDXL-Turbo (lazy pipeline loading) |
| Demo | Generated and displayed multiple images inline |
| GPU Monitor | VRAM usage during diffusion visible |
| Registration | `AIStudio-EQ-ImageGen` registered in Model Registry |
| Verification | Loaded model generated test image |

## Next Steps

- **Explore another capability:** Open `document-analyzer-starter.ipynb`
- **Deploy in AI Studio:** Select `AIStudio-EQ-ImageGen` → click Deploy → use the image gen Streamlit UI
- **Try different prompts:** Experiment with style keywords: `photorealistic`, `oil painting`, `watercolor`, `digital art`
- **Adjust quality vs. speed:** Edit `num_inference_steps` in `ImageGenModel._generate()` (1–8 steps)